# DATA 612 – Project 1  
## Global Baseline Predictors and RMSE

**Author:** Kevin Martin  
**Course:** DATA 612 – Recommender Systems  
**Program:** MS Data Science, CUNY School of Professional Studies  
**Date:** June 2026


## Introduction

In this project, I will build a simple recommender system using a small toy dataset. The goal is to predict user ratings with very little information. First, I will use a raw average rating as the simplest prediction. Then, I will improve the prediction by accounting for user bias and item bias.

This project is focused on understanding the basic logic behind recommender systems before moving into more advanced methods later in the course.


## Business Perspective

The recommender system I am building is for a small streaming platform that recommends movies and shows to users. Users rate content on a scale from 1 to 5, where 1 means they did not like the item and 5 means they liked it very much.

The business goal is to use past ratings to make better predictions about how users may rate movies or shows they have not watched yet. Even though this is a small toy dataset, the same general idea can apply to larger platforms like Netflix, YouTube, Amazon Prime Video, or Hulu.


## Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error

## Create a Toy Ratings Dataset

The assignment requires at least five users, at least five items, numeric ratings, and some missing data. I created a small movie/show ratings dataset with six users and six items.

Missing values represent user-item combinations where the user has not rated the item.


In [2]:
ratings_matrix = pd.DataFrame({
    "Bridgerton": [5, 4, np.nan, 5, 3, 4],
    "The Crown": [5, 5, 4, np.nan, 3, 4],
    "Heartstopper": [4, 5, 5, 4, np.nan, 5],
    "The Witcher": [2, 3, 4, np.nan, 5, 3],
    "Wednesday": [5, 4, np.nan, 5, 4, 4],
    "Stranger Things": [4, np.nan, 5, 4, 5, 4]
}, index=["Kevin", "Robert", "Cody", "Mary", "John", "Luis"])

ratings_matrix

,Bridgerton,The Crown,Heartstopper,The Witcher,Wednesday,Stranger Things
Kevin,5.0,5.0,4.0,2.0,5.0,4.0
Robert,4.0,5.0,5.0,3.0,4.0,NaN
Cody,NaN,4.0,5.0,4.0,NaN,5.0
Mary,5.0,NaN,4.0,NaN,5.0,4.0
John,3.0,3.0,NaN,5.0,4.0,5.0
Luis,4.0,4.0,5.0,3.0,4.0,4.0


## Convert the Matrix to Long Format

A user-item matrix is helpful for viewing the data, but a long-format table is easier for splitting the ratings into training and test datasets.


In [3]:
ratings_long = ratings_matrix.reset_index().melt(
    id_vars="index",
    var_name="item",
    value_name="rating"
).rename(columns={"index": "user"})

ratings_long = ratings_long.dropna().reset_index(drop=True)

ratings_long.head(10)

,user,item,rating
0,Kevin,Bridgerton,5.0
1,Robert,Bridgerton,4.0
2,Mary,Bridgerton,5.0
3,John,Bridgerton,3.0
4,Luis,Bridgerton,4.0
5,Kevin,The Crown,5.0
6,Robert,The Crown,5.0
7,Cody,The Crown,4.0
8,John,The Crown,3.0
9,Luis,The Crown,4.0


## Split Ratings into Training and Test Sets

The training set is used to calculate the global average, user bias, and item bias. The test set is used to check how well the model performs on ratings it did not use for training.

I used a random split, with 80% of the observed ratings for training and 20% for testing.


In [4]:
np.random.seed(42)

ratings_long["set"] = np.where(
    np.random.rand(len(ratings_long)) < 0.8,
    "train",
    "test"
)

train = ratings_long[ratings_long["set"] == "train"].copy()
test = ratings_long[ratings_long["set"] == "test"].copy()

print("Training ratings:", len(train))
print("Test ratings:", len(test))

train.head()

Training ratings: 26
Test ratings: 4


,user,item,rating,set
0,Kevin,Bridgerton,5.0,train
2,Mary,Bridgerton,5.0,train
3,John,Bridgerton,3.0,train
4,Luis,Bridgerton,4.0,train
5,Kevin,The Crown,5.0,train


## Raw Average Predictor

The simplest possible recommender model predicts the same rating for every user-item combination. This prediction is the average rating from the training data.

This is called the global average or raw average predictor.


In [5]:
global_mean = train["rating"].mean()
global_mean

np.float64(4.153846153846154)

## RMSE for Raw Average Predictor

RMSE stands for Root Mean Squared Error. It measures how far the predicted ratings are from the actual ratings. A lower RMSE means the model is making better predictions.


In [6]:
train["raw_average_prediction"] = global_mean
test["raw_average_prediction"] = global_mean

rmse_train_raw = np.sqrt(mean_squared_error(train["rating"], train["raw_average_prediction"]))
rmse_test_raw = np.sqrt(mean_squared_error(test["rating"], test["raw_average_prediction"]))

print("Training RMSE - Raw Average:", round(rmse_train_raw, 4))
print("Test RMSE - Raw Average:", round(rmse_test_raw, 4))

Training RMSE - Raw Average: 0.8177
Test RMSE - Raw Average: 0.6081


## Calculate User Bias

User bias accounts for the fact that some users tend to rate items higher or lower than average.

For example, one user may be very generous and give mostly 5-star ratings, while another user may be more critical and give lower ratings.


In [7]:
user_bias = train.groupby("user")["rating"].mean() - global_mean
user_bias = user_bias.rename("user_bias")

user_bias

,user_bias
user,
Cody,0.346154
John,-0.153846
Kevin,0.012821
Luis,-0.153846
Mary,0.346154
Robert,-0.153846


## Calculate Item Bias

Item bias accounts for the fact that some movies or shows tend to receive higher or lower ratings overall.

For example, a popular show may receive higher ratings across many users, while another item may receive lower ratings.


In [8]:
item_bias = train.groupby("item")["rating"].mean() - global_mean
item_bias = item_bias.rename("item_bias")

item_bias

,item_bias
item,
Bridgerton,0.096154
Heartstopper,0.179487
Stranger Things,0.246154
The Crown,0.096154
The Witcher,-0.753846
Wednesday,0.246154


## Baseline Predictor

The baseline predictor improves the raw average prediction by adding user bias and item bias.

The formula is:

**Baseline Prediction = Global Mean + User Bias + Item Bias**

This still does not use advanced recommendation methods, but it is better than predicting the same average for everyone.


In [9]:
def add_bias_predictions(data, global_mean, user_bias, item_bias):
    data = data.copy()
    data = data.merge(user_bias, on="user", how="left")
    data = data.merge(item_bias, on="item", how="left")

    # If a user or item appears in the test set but not in the training set,
    # fill the missing bias with 0.
    data["user_bias"] = data["user_bias"].fillna(0)
    data["item_bias"] = data["item_bias"].fillna(0)

    data["baseline_prediction"] = (
        global_mean + data["user_bias"] + data["item_bias"]
    )

    # Ratings are on a 1 to 5 scale, so predictions should stay within that range.
    data["baseline_prediction"] = data["baseline_prediction"].clip(1, 5)

    return data

train_predictions = add_bias_predictions(train, global_mean, user_bias, item_bias)
test_predictions = add_bias_predictions(test, global_mean, user_bias, item_bias)

train_predictions.head()

,user,item,rating,set,raw_average_prediction,user_bias,item_bias,baseline_prediction
0,Kevin,Bridgerton,5.0,train,4.153846,0.012821,0.096154,4.262821
1,Mary,Bridgerton,5.0,train,4.153846,0.346154,0.096154,4.596154
2,John,Bridgerton,3.0,train,4.153846,-0.153846,0.096154,4.096154
3,Luis,Bridgerton,4.0,train,4.153846,-0.153846,0.096154,4.096154
4,Kevin,The Crown,5.0,train,4.153846,0.012821,0.096154,4.262821


## RMSE for Baseline Predictor

In [10]:
rmse_train_baseline = np.sqrt(mean_squared_error(
    train_predictions["rating"],
    train_predictions["baseline_prediction"]
))

rmse_test_baseline = np.sqrt(mean_squared_error(
    test_predictions["rating"],
    test_predictions["baseline_prediction"]
))

print("Training RMSE - Baseline Predictor:", round(rmse_train_baseline, 4))
print("Test RMSE - Baseline Predictor:", round(rmse_test_baseline, 4))

Training RMSE - Baseline Predictor: 0.7074
Test RMSE - Baseline Predictor: 0.534


## Compare Results

In [11]:
results = pd.DataFrame({
    "Model": ["Raw Average", "Baseline Predictor"],
    "Training RMSE": [rmse_train_raw, rmse_train_baseline],
    "Test RMSE": [rmse_test_raw, rmse_test_baseline]
})

results

,Model,Training RMSE,Test RMSE
0,Raw Average,0.817704,0.608130
1,Baseline Predictor,0.707444,0.533998


## Predicted Ratings Table

The table below shows the actual ratings and baseline predictions for the test data.


In [12]:
test_predictions[["user", "item", "rating", "baseline_prediction"]]

,user,item,rating,baseline_prediction
0,Robert,Bridgerton,4.0,4.096154
1,Cody,The Crown,4.0,4.596154
2,Robert,Heartstopper,5.0,4.179487
3,Cody,Heartstopper,5.0,4.679487


## Summary of Results

The raw average predictor is the simplest possible model because it predicts the same rating for every user-item combination. This gives us a basic starting point, but it does not account for the fact that some users rate higher or lower than others, and some items are more popular than others.

The baseline predictor improves on this by adding user bias and item bias. User bias adjusts for each user's rating behavior, while item bias adjusts for how each movie or show is rated overall.

In this project, the baseline predictor performed better on the training data because it used more information from the training set. The test RMSE may improve or may not improve depending on the small random test sample, but the baseline method is still conceptually better because it makes predictions that are specific to the user and item.

This project helped me understand how recommender systems can start with simple averages and gradually become more personalized. Even before using more advanced methods like collaborative filtering or matrix factorization, accounting for user and item bias can make the predictions more meaningful.


## AI Use Statement

Generative AI was used to assist with brainstorming, organization, explanation, and editing. All content was reviewed, revised, and validated by the author.
